# Course 8: Research Engineering Interviews
## Session 9: ML system design

**Goal:** lead two 40-minute design rounds and back the design with estimates.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** the tests and reference solutions here were run once on a CPU (November 2026). Cells marked TODO are stubs for you; try them under a timer before opening the reference solutions at the bottom.

### Prompts
1. Design a nightly evaluation system for a language model.
2. Design a pipeline that builds a deduplicated pretraining corpus from web text.
3. Design an experiment tracker for a 20-person research team.
4. Design a fine-tuning service for customers' private data.

### First five minutes: questions to ask
Purpose and users | definition of success | scale | constraints (latency, budget, hardware, privacy)

### Estimate helpers

In [ ]:
import math
def eval_minutes(prompts, tokens_each, tokens_per_s_per_gpu, gpus=1):
    return prompts * tokens_each / (tokens_per_s_per_gpu * gpus) / 60

print("10,000 prompts x 500 tokens at 2,000 tok/s on 1 GPU:", round(eval_minutes(10_000, 500, 2000), 1), "minutes")
need = 10_000 * 500 / 2000 / (5 * 60)
print("GPUs to finish in 5 minutes:", round(need, 1), "->", math.ceil(need))

def storage_tb(docs, bytes_each): return docs * bytes_each / 1e12
print("1 billion web pages at 50 KB:", storage_tb(1e9, 50e3), "TB")

### Is a difference real?
A nightly system should report uncertainty beside every score.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
old = rng.random(2000) < 0.700          # per-prompt correctness of yesterday's model (simulated)
new = rng.random(2000) < 0.703          # today's model, truly 0.3 points better
diffs = []
for _ in range(2000):
    i = rng.integers(0, 2000, 2000)
    diffs.append(new[i].mean() - old[i].mean())
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"observed difference {new.mean() - old.mean():+.3f}, 95% interval [{lo:+.3f}, {hi:+.3f}]")

With 2,000 prompts the interval is several points wide, so a 0.3-point change cannot be told from noise. Say that in the design, and size the evaluation set accordingly.

### Checklist for the interviewer
- Asked about purpose, scale and constraints
- Stated assumptions and wrote them down
- Proposed a simple baseline first
- Covered data, training, evaluation, serving
- Gave at least three numerical estimates
- Named two trade-offs and chose
- Named the main failure mode and a monitor
- Left time to summarize

### Write-up (TODO)
One page: diagram, assumptions, three estimates, two trade-offs, main risk.